# Schema Compatibility Audit for `008_accounting_foundation.sql`

This notebook inspects the live `sacco_system` database and evaluates compatibility for tables referenced by the new accounting migration.

Goals:
- Confirm existence and structure of `users`, `members`, `ledger_entries`, `shares`, `loans`, `savings_accounts`, and migration-referenced tables.
- Build a foreign key compatibility matrix for the migration.
- Check journal entry number uniqueness, `account_code` length consistency, and `bank_account_id` references.
- Report whether the migration would fail against the current live schema.


In [ ]:
import json
import re
import subprocess
from pathlib import Path

workspace_root = Path(r"d:/wamp64/www/sacco")
php_binary = "php"

relevant_tables = [
    'users',
    'members',
    'ledger_entries',
    'shares',
    'loans',
    'savings_accounts',
    'chart_of_accounts',
    'journal_entries',
    'journal_entry_lines',
    'bank_accounts',
    'cash_book'
]

migration_path = workspace_root / 'migrations' / '008_accounting_foundation.sql'
print(f"Workspace root: {workspace_root}")
print(f"Migration file: {migration_path}")
print(f"PHP binary: {php_binary}")


In [ ]:
def fetch_live_table_metadata(tables):
    php_code = r"""
require 'config/db_connection.php';
$db = getDB();
$tables = %s;
$out = [];
foreach ($tables as $table) {
    $stmt = $db->prepare('SELECT TABLE_NAME, ENGINE FROM information_schema.tables WHERE table_schema = DATABASE() AND TABLE_NAME = ?');
    $stmt->execute([$table]);
    $tableInfo = $stmt->fetch(PDO::FETCH_ASSOC);
    if (!$tableInfo) {
        $out[$table] = ['exists' => false];
        continue;
    }
    $stmt2 = $db->prepare('SELECT COLUMN_NAME, COLUMN_TYPE, DATA_TYPE, IS_NULLABLE, COLUMN_KEY, EXTRA FROM information_schema.columns WHERE table_schema = DATABASE() AND table_name = ? ORDER BY ORDINAL_POSITION');
    $stmt2->execute([$table]);
    $cols = $stmt2->fetchAll(PDO::FETCH_ASSOC);
    $pk = [];
    foreach ($cols as $col) {
        if ($col['COLUMN_KEY'] === 'PRI') {
            $pk[] = $col['COLUMN_NAME'];
        }
    }
    $out[$table] = [
        'exists' => true,
        'engine' => $tableInfo['ENGINE'],
        'columns' => $cols,
        'primary_key' => $pk,
    ];
}
echo json_encode($out, JSON_PRETTY_PRINT);
""" % json.dumps(tables)

    result = subprocess.run(
        [php_binary, '-r', php_code],
        cwd=str(workspace_root),
        capture_output=True,
        text=True,
        check=True,
    )
    return json.loads(result.stdout)

live_metadata = fetch_live_table_metadata(relevant_tables)
print(json.dumps(live_metadata, indent=2))


In [ ]:
migration_sql = migration_path.read_text(encoding='utf-8')

fk_pattern = re.compile(
    r'FOREIGN KEY \((?P<col>[^)]+)\) REFERENCES (?P<table>\w+)\((?P<refcol>[^)]+)\)',
    re.IGNORECASE
)
foreign_keys = []
for match in fk_pattern.finditer(migration_sql):
    foreign_keys.append({
        'table': match.group(0),
        'column': match.group('col').strip(),
        'referenced_table': match.group('table').strip(),
        'referenced_column': match.group('refcol').strip(),
    })

print('Foreign keys found:')
for fk in foreign_keys:
    print(fk)

referenced_tables = sorted({fk['referenced_table'] for fk in foreign_keys} | set(relevant_tables))
print('\nReferenced tables:')
print(referenced_tables)


In [ ]:
def classify_foreign_key(fk, metadata):
    table = fk['referenced_table']
    col = fk['referenced_column']
    if table not in metadata or not metadata[table].get('exists'):
        return 'TABLE MISSING'
    table_meta = metadata[table]
    if table_meta['engine'].upper() != 'INNODB':
        return 'ENGINE INCOMPATIBLE'
    col_meta = next((c for c in table_meta['columns'] if c['COLUMN_NAME'] == col), None)
    if col_meta is None:
        return 'COLUMN MISSING'
    return 'VALID'

fk_matrix = []
for fk in foreign_keys:
    status = classify_foreign_key(fk, live_metadata)
    fk_matrix.append({**fk, 'status': status})

print('Foreign key compatibility matrix:')
for row in fk_matrix:
    print(row)


def find_column(table, name):
    t = live_metadata.get(table)
    if not t or not t.get('exists'):
        return None
    return next((c for c in t['columns'] if c['COLUMN_NAME'] == name), None)

journal_ref_col = find_column('journal_entries', 'reference_number')
journal_ref_unique = None
stats = []
if journal_ref_col is not None:
    php_code = r"""
require 'config/db_connection.php';
$db = getDB();
$stmt = $db->prepare('SELECT NON_UNIQUE, INDEX_NAME FROM information_schema.statistics WHERE table_schema = DATABASE() AND table_name = ? AND column_name = ?');
$stmt->execute(['journal_entries', 'reference_number']);
$rows = $stmt->fetchAll(PDO::FETCH_ASSOC);
echo json_encode($rows);
"""
    result = subprocess.run([php_binary, '-r', php_code], cwd=str(workspace_root), capture_output=True, text=True, check=True)
    stats = json.loads(result.stdout)
    journal_ref_unique = any(r['NON_UNIQUE'] == 0 for r in stats)

account_code_lengths = {}
for table in ['chart_of_accounts', 'journal_entry_lines']:
    col = find_column(table, 'account_code')
    account_code_lengths[table] = col['COLUMN_TYPE'] if col else None
account_code_length_matches = account_code_lengths['chart_of_accounts'] == account_code_lengths['journal_entry_lines']

bank_account_id_ref = find_column('bank_accounts', 'bank_account_id')
bank_account_id_valid = bank_account_id_ref is not None and bank_account_id_ref['COLUMN_KEY'] == 'PRI' and bank_account_id_ref['COLUMN_TYPE'].upper().startswith('BIGINT')

print('\nJournal entry reference uniqueness check:')
print('journal_entries.reference_number exists:', journal_ref_col is not None)
print('journal_entries.reference_number unique index present:', journal_ref_unique)
print('index stats:', stats)

print('\naccount_code length/types:')
print(json.dumps(account_code_lengths, indent=2))
print('account_code length match:', account_code_length_matches)

print('\nbank_account_id reference validity:')
print('bank_accounts.bank_account_id exists:', bank_account_id_ref is not None)
print('bank_accounts.bank_account_id primary key:', bank_account_id_ref['COLUMN_KEY'] == 'PRI' if bank_account_id_ref else None)
print('bank_accounts.bank_account_id type:', bank_account_id_ref['COLUMN_TYPE'] if bank_account_id_ref else None)
